# Step 02: seeds → graph

Each source grows the seeds from its curated facts, as `[expand.<source>]` asks:

- **MeSH tree** around disease seeds: ancestors, relatives (siblings and their subtypes), descendants
- **supplementary records** filed under those headings (rare diseases)
- **see-also** links, and for drug seeds their **drug classes** and the class's other drugs
- **HGNC gene families** for gene seeds

Every node has a `distance` (steps from the nearest seed) and a `reason` (how it got in); step 03 picks topics from them. Each round gets its own `graph_<n>`.

All the example notebooks use `experiments/examples.toml`: the default experiment with tiny limits and its own pool (`corpora/examples`). Run `00-setup` for it first: `python src/00-setup/run.py --experiment experiments/examples.toml --no-index`.

In [1]:
import importlib.util
import sys
import tomllib
from pathlib import Path

import duckdb
import pyarrow as pa

SRC = Path.cwd().parent
LACUNA = SRC.parent
ROOT = LACUNA / 'data'
experiment = tomllib.loads((LACUNA / 'experiments' / 'examples.toml').read_text())


def load(step):
    """A step's run.py as a module (every step's entry point is called run.py)."""
    sys.path.insert(0, str(SRC / step))
    spec = importlib.util.spec_from_file_location(f'step_{step[:2]}', SRC / step / 'run.py')
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


query, expand = load('01-query'), load('02-expand')


def show(goal):
    seeds = query.seeds(goal, experiment, ROOT)
    nodes, edges = expand.build([s['id'] for s in seeds], experiment, ROOT)
    print(f'{goal!r}: {nodes.num_rows} nodes, {edges.num_rows} edges from {[s["id"] for s in seeds]}')
    print(duckdb.sql('SELECT reason, count(*) AS nodes FROM nodes GROUP BY reason ORDER BY min(distance), nodes DESC'))
    return nodes, edges

## 1. One disease

Glioblastoma is Astrocytoma's only child in MeSH, so one *branching* level up is Glioma: its other subtypes come in as relatives.

In [2]:
nodes, edges = show('new treatment avenues for glioblastoma')
print(duckdb.sql('SELECT id, name, distance, reason FROM nodes ORDER BY distance, name'))

'new treatment avenues for glioblastoma': 17 nodes, 16 edges from ['mesh:D005909']
┌───────────────┬───────┐
│    reason     │ nodes │
│    varchar    │ int64 │
├───────────────┼───────┤
│ seed          │     1 │
│ ancestor      │     2 │
│ supplementary │     6 │
│ relative      │     8 │
└───────────────┴───────┘

┌─────────────────┬──────────────────────────────────────────────┬──────────┬───────────────┐
│       id        │                     name                     │ distance │    reason     │
│     varchar     │                   varchar                    │  int64   │    varchar    │
├─────────────────┼──────────────────────────────────────────────┼──────────┼───────────────┤
│ mesh:D005909    │ Glioblastoma                                 │        0 │ seed          │
│ mesh:D001254    │ Astrocytoma                                  │        1 │ ancestor      │
│ mesh:C535354    │ Cerebral astrocytoma, adult                  │        2 │ supplementary │
│ mesh:C535945    │ Desm

## 2. A drug and a disease

Drugs aren't grown through the chemical tree (it groups by structure). They come in through their drug classes, and a small class brings its other drugs: other alkylating agents for temozolomide.

In [3]:
nodes, edges = show('temozolomide resistance in glioblastoma')
print(duckdb.sql("SELECT name, distance, reason FROM nodes WHERE reason IN ('drug_class', 'same_drug_class') "
                 "AND length(name) < 30 ORDER BY distance, name"))

'temozolomide resistance in glioblastoma': 109 nodes, 169 edges from ['mesh:D000077204', 'mesh:D005909']
┌─────────────────┬───────┐
│     reason      │ nodes │
│     varchar     │ int64 │
├─────────────────┼───────┤
│ seed            │     2 │
│ drug_class      │     3 │
│ ancestor        │     2 │
│ same_drug_class │    88 │
│ supplementary   │     6 │
│ relative        │     8 │
└─────────────────┴───────┘

┌────────────────────────────┬──────────┬─────────────────┐
│            name            │ distance │     reason      │
│          varchar           │  int64   │     varchar     │
├────────────────────────────┼──────────┼─────────────────┤
│ Alkylating Agents          │        1 │ drug_class      │
│ Antineoplastic Agents      │        1 │ drug_class      │
│ 1,2-Dimethylhydrazine      │        2 │ same_drug_class │
│ 3-(2-cyanoethyl)thymidine  │        2 │ same_drug_class │
│ Acetoxyacetylaminofluorene │        2 │ same_drug_class │
│ Altretamine                │        2 │ same

## 3. A gene and a disease

The gene brings its family; the disease its MeSH neighbours and see-also links.

In [4]:
nodes, edges = show('EGFR inhibitors for non-small cell lung cancer')
print(duckdb.sql('SELECT id, name, types, distance, reason FROM nodes ORDER BY distance, name'))
print(duckdb.sql('SELECT s, r, o, layer, source FROM edges'))

'EGFR inhibitors for non-small cell lung cancer': 12 nodes, 10 edges from ['ncbigene:1956', 'mesh:D002289']
┌───────────────────┬───────┐
│      reason       │ nodes │
│      varchar      │ int64 │
├───────────────────┼───────┤
│ seed              │     2 │
│ see_also          │     4 │
│ gene_group        │     1 │
│ ancestor          │     1 │
│ gene_group_member │     3 │
│ relative          │     1 │
└───────────────────┴───────┘

┌─────────────────┬──────────────────────────────────┬──────────────┬──────────┬───────────────────┐
│       id        │               name               │    types     │ distance │      reason       │
│     varchar     │             varchar              │  varchar[]   │  int64   │      varchar      │
├─────────────────┼──────────────────────────────────┼──────────────┼──────────┼───────────────────┤
│ mesh:D002289    │ Carcinoma, Non-Small-Cell Lung   │ [disease]    │        0 │ seed              │
│ ncbigene:1956   │ EGFR                             │ [

## From the command line

```
python src/02-expand/run.py --experiment experiments/default.toml
```
reads `runs/<experiment>/01-query/seeds.json` and writes `runs/<experiment>/02-expand/graph_<n>/` (`nodes.parquet`, `edges.parquet`), where n is the next topic round.